## GPT-2 style language model training

This project trains a small GPT-2 style language model from a text file using PyTorch and HuggingFace Transformers. The training is intentionally lightweight training for demo purposes. Dockerfile and script are available in the mini-gpt directory. Docker image available here: quay.io/amscesnet/minigpt:dev

The workflow is: Large Text file -> Take first 1000 lines -> Train BPE tokenizer -> Tokenize dataset -> Train tiny GPT-2 model -> Save artifacts + generate sample text.

Once the workflow finishes, download the sample text file.

## Prerequisites

### Required Packages
- `amscrot-py` (installed)
- `globus-sdk` (installed)
- `dotenv` (installed)

### Installation

```
pip install amscrot-py globus-sdk dotenv
```

### Credentials

IRI service clients authenticate via API keys stored in `~/.amscrot/credentials.yml`. You need entries for each site that will be used for Job submission.

```yaml
# ~/.amscrot/credentials.yml

esnet-iri-east:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://iri-dev.ppg.es.net

esnet-iri-west:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://esnet-west.sdn-sense.net

nersc-iri:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://api.iri.nersc.gov
```

---
## 0. Retrieve an authentication token

The process of issuing AmSC tokens is evolving rapidly. For IRI API usage, see the following repository for examples on how to retrieve a token for use with this toolkit.

  * https://github.com/doe-iri/iri-facility-api-examples

Once you have a token, copy it to a `.env` file, set in the `AMSC_TOKEN` environment variable, or paste into the following cell to generate a new credential file for this toolkit example.

In [ ]:
import os
import yaml
from dotenv import load_dotenv

AMSC_TOKEN = None   # <-- manually set token
if not AMSC_TOKEN:
    load_dotenv()   # take environment variables from .env file (if present)
    AMSC_TOKEN = os.getenv("AMSC_TOKEN")

# Create credentials data
credentials = {
    'esnet-iri-east': {
        'api_key': AMSC_TOKEN,
        'api_endpoint': 'https://iri-dev.ppg.es.net'
    },
    'esnet-iri-west': {
        'api_key': AMSC_TOKEN,
        'api_endpoint': 'https://esnet-west.sdn-sense.net'
    },
    'nersc-iri': {
        'api_key': AMSC_TOKEN,
        'api_endpoint': 'https://api.iri.nersc.gov'
    }
}

# Write to ~/.amscrot/credentials-new.yml
cred_path = os.path.expanduser("~/.amscrot/credentials-new.yml")
os.makedirs(os.path.dirname(cred_path), exist_ok=True)
with open(cred_path, 'w') as f:
    yaml.dump(credentials, f)
print(f"Wrote credentials to {cred_path}")

---
## 1. Initialize Client & Session

In [ ]:
import time
from amscrot.client.client import Client
from amscrot.client.job import Job, JobType, JobServiceType, JobSpec, JobState
from amscrot.serviceclient import ServiceClient
from amscrot.util.constants import Constants

client = Client()
session = client.create_session("gpt2-model-train")
print("Client and session initialized.")

## 3. Set Up IRI Service Clients

We create three service clients — one for each IRI site. Each client loads its credentials from the corresponding profile in `~/.amscrot/credentials-new.yml`.

For this training job, we will select just a single site's compute resource.

In [ ]:
east_client = ServiceClient.create(
    type=Constants.ServiceType.AMSC_IRI,
    name="iri-east",
    profile="esnet-iri-east",
    credential_file="~/.amscrot/credentials-new.yml"
)
session.add_service_client(east_client)

west_client = ServiceClient.create(
    type=Constants.ServiceType.AMSC_IRI,
    name="iri-west",
    profile="esnet-iri-west",
    credential_file="~/.amscrot/credentials-new.yml"
)
session.add_service_client(west_client)

nersc_client = ServiceClient.create(
    type=Constants.ServiceType.AMSC_IRI,
    name="nersc-iri",
    profile="nersc-iri",
    credential_file="~/.amscrot/credentials-new.yml"
)
session.add_service_client(nersc_client)

## 4. Discover Compute Resources

Each service client's `discover()` method returns a `DiscoveryResult` container with typed accessors for each resource type. We use `.compute` to find available compute resources at each site.

In [ ]:
east_discovery = east_client.discover()
west_discovery = west_client.discover()
nersc_discovery = nersc_client.discover()

print(f"ESnet East discovery: {east_discovery.summary()}")
print(f"ESnet West discovery: {west_discovery.summary()}")
print(f"NERSC discovery: {nersc_discovery.summary()}")

assert east_discovery.compute, "No compute resources found on East site!"
assert west_discovery.compute, "No compute resources found on West site!"
assert nersc_discovery.compute, "No compute resources found on NERSC site!"

# For ESnet, just grab the first compute resource
east_resource_id = east_discovery.compute[0].data.get("id")
west_resource_id = west_discovery.compute[0].data.get("id")
compute_resources = nersc_discovery.compute

# For NERSC, locate the compute resource with group "perlmutter" and named "compute"
target_resource = None
for resource in compute_resources:
    data = resource.data
    if data.get('group') == 'perlmutter' and data.get('name') == 'compute':
        target_resource = resource
        break

if not target_resource:
    self.skipTest("Target compute resource (perlmutter/compute) not found.")

nersc_resource_data = target_resource.data
nersc_resource_id = nersc_resource_data.get('id')

if nersc_resource_id:
    print(f"Found target NERSC compute resource: {nersc_resource_data}")

print(f"\nEast resource_id: {east_resource_id}")
print(f"West resource_id: {west_resource_id}")
print(f"NERSC resource_id: {nersc_resource_id}")

## 5. Define Job Specs & Jobs

Prepare model training bash script as JobSpec attributes and submit Job.

The `resource_id` is set dynamically from the discovery step above.

In [ ]:
import os
import datetime as dt

DEFAULT_JOB_DIR = os.environ.get("IRI_JOB_DIR", "/data/home/kissel") # <-- adjust path

# -----------------------------------------------------------
# Paths
# -----------------------------------------------------------
timestamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%d-%H%M%S")

input_file = f"{DEFAULT_JOB_DIR}/synthetic_training_data_1g.txt"
output_dir = f"{DEFAULT_JOB_DIR}/amsc-iri-demo-results-{timestamp}"

stdout_path = f"{DEFAULT_JOB_DIR}/minigpt_stdout_{timestamp}.log"
stderr_path = f"{DEFAULT_JOB_DIR}/minigpt_stderr_{timestamp}.log"

# -----------------------------------------------------------
# Training parameters
# -----------------------------------------------------------
image = "quay.io/amscesnet/minigpt:dev"

num_steps = 1000
num_lines = 1000000

# -----------------------------------------------------------
# Bash payload executed on compute node
# -----------------------------------------------------------
bash_payload = f"""
set -euo pipefail

echo "=== MiniGPT training job ==="
echo "UTC now: $(date -u '+%Y-%m-%dT%H:%M:%SZ')"
echo "Hostname: $(hostname)"

echo "Input file: {input_file}"
echo "Output dir: {output_dir}"

mkdir -p "{output_dir}"

echo "Starting training..."

python3 /root/tiny_gpt2_cpu_1k.py \
    "{input_file}" \
    "{output_dir}" \
    {num_steps} \
    {num_lines}

echo "Training finished."

echo "Output directory contents:"
ls -lah "{output_dir}"
"""

common_resources = {
    "node_count": 1,
    "process_count": 1,
    "processes_per_node": 1,
    "cpu_cores_per_process": 4,
    "gpu_cores_per_process": None,
    "exclusive_node_use": False,
    "memory": 4000000000
}

esnet_attributes = {
    "container": {"image": image},
    "inherit_environment": True,
    "directory": DEFAULT_JOB_DIR,
    "duration": 7200,
    "queue_name": "debug",
    "account": "interactive",
    "stdout_path": stdout_path,
    "stderr_path": stderr_path
}

nersc_attributes = {
    "container": {"image": image},
    "inherit_environment": True,
    "duration": 7200,
    "queue_name": "debug",
    "account": "amsc013",
    "stdout_path": "stdout.log",
    "stderr_path": "stderr.log"

}

spec_east = JobSpec(
    executable="bash",
    arguments=["-lc", bash_payload],
    resources=common_resources,
    attributes={"resource_id": east_resource_id, **esnet_attributes}
)

spec_west = JobSpec(
    executable="bash",
    arguments=["-lc", bash_payload],
    resources=common_resources,
    attributes={"resource_id": west_resource_id, **nersc_attributes}
)

spec_nersc = JobSpec(
    executable="bash",
    arguments=["-lc", bash_payload],
    resources=common_resources,
    attributes={"resource_id": nersc_resource_id, **nersc_attributes}
)

job1 = Job(name="job-1", type=JobType.COMPUTE,
            service_type=JobServiceType.BATCH,
            service_client=east_client,
            job_spec=spec_east)

job2 = Job(name="job-2", type=JobType.COMPUTE,
            service_type=JobServiceType.BATCH,
            service_client=west_client,
            job_spec=spec_west)

job3 = Job(name="job-3", type=JobType.COMPUTE,
            service_type=JobServiceType.BATCH,
            service_client=nersc_client,
            job_spec=spec_nersc)

session.add_job(job1)
# session.add_job(job2)
# session.add_job(job3)

print("Jobs defined and added to session.")

## 6. Plan

The plan phase validates all resources and job specs before anything is created.

In [ ]:
try:
    session.plan()
except Exception as e:
    print(f"Failed to plan jobs: {e}")
session.show()

## 7. Apply and monitor

Apply creates any resources (if applicable) and submits the defined compute jobs.
`session.wait()` then polls both jobs until they complete (or raise `WaitTimeoutError`).

In [ ]:
try:
    rc = session.apply()
    print("Session applied successfully.")
except Exception as e:
    print(f"Failed to apply session: {e}")
    raise

results = session.wait(
    # jobs=[job1, job2, job3],
    jobs = [job1],
    target_states=[JobState.COMPLETED, JobState.FAILED, JobState.CANCELED],
    timeout=7200,
    interval=2,
    verbose=True,
)

s1 = results["job-1"]
# s2 = results["job-2"]
# s3 = results["job-3"]

assert s1.state == JobState.COMPLETED, f"Job1 failed or timed out: {s1}"
# assert s2.state == JobState.COMPLETED, f"Job2 failed or timed out: {s2}"
# assert s3.state == JobState.COMPLETED, f"Job3 failed or timed out: {s3}"

print(f"\n✅ All jobs completed successfully!")

## 8. View job logs

Fetch files from the Session and display stdout logs.

In [ ]:
fetched = session.fetch_output_files(jobs=[job1])
print(f"Fetched files: {fetched}")

for job_name, paths in fetched.items():
    for k,path in paths.items():
        print(f"\n--- {k} for {job_name} ---")
        try:
            with open(path) as f:
                print(f.read())
        except Exception as e:
            print(f"  (could not read {k}: {e})")

## 9. Clean Up

Destroy the session to tear down any provisioned resources and cancel remaining jobs.

In [ ]:
session.destroy()